In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path


In [2]:
input_path = Path("../data/processed/products_prepared.csv")

if not input_path.exists():
    raise FileNotFoundError("Exécute d'abord le notebook 02_data_preprocessing.ipynb.")

df = pd.read_csv(input_path, encoding="utf-8-sig")

print("Dimensions du dataset :", df.shape)
display(df.head(3))


Dimensions du dataset : (9, 24)


,product_id,name,domain,category,subcategory,brand,model,description,material,diameter_mm,...,voltage_v,dimensions_mm,unit,price_dzd,availability,image_url,source_id,last_verified,notes,content_text
0,SH001,Chauffe-Bain 11L Junkers,Chauffage,Chaudière / générateur,Chauffe-bain gaz,JUNKERS,NaN,Chauffe-bain 11 L ; la fiche consultée indique...,NaN,NaN,...,NaN,NaN,pièce,28500.0,In stock,NaN,SRC001,2026-11-13,Prix et stock affichés par le vendeur au momen...,Chauffe-Bain 11L Junkers Chauffage Chaudière /...
1,SH002,Chauffe-eau Smart 12L ENERGICAL,Chauffage,Chaudière / générateur,Chauffe-eau gaz,ENERGICAL,Smart 12L,Chauffe-eau 12 L ; variantes de couleur affich...,NaN,NaN,...,NaN,NaN,pièce,29500.0,In stock,NaN,SRC001,2026-11-13,Prix et stock affichés par le vendeur au momen...,Chauffe-eau Smart 12L ENERGICAL Chauffage Chau...
2,SH003,Chaudière ventouse 28 kW Energical,Chauffage,Chaudière / générateur,Chaudière gaz,ENERGICAL,28 kW ventouse,Chaudière ventouse annoncée avec puissance nom...,NaN,NaN,...,NaN,NaN,pièce,NaN,NaN,NaN,SRC002,2026-11-13,La puissance provient du titre catalogue ; pri...,Chaudière ventouse 28 kW Energical Chauffage C...


In [3]:
text_columns = [
    "name", "domain", "category", "subcategory",
    "brand", "model", "description", "material", "connection_type"
]

numeric_columns = [
    "diameter_mm", "length_m", "pressure_rating_bar",
    "power_kw", "voltage_v", "price_dzd"
]

text_columns = [col for col in text_columns if col in df.columns]
numeric_columns = [col for col in numeric_columns if col in df.columns]

print("Caractéristiques textuelles :", text_columns)
print("Caractéristiques numériques :", numeric_columns)


Caractéristiques textuelles : ['name', 'domain', 'category', 'subcategory', 'brand', 'model', 'description', 'material', 'connection_type']
Caractéristiques numériques : ['diameter_mm', 'length_m', 'pressure_rating_bar', 'power_kw', 'voltage_v', 'price_dzd']


In [4]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).lower().strip()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\\s+", " ", text)

    return text


In [5]:
df_features = df.copy()

for col in text_columns:
    df_features[col] = df_features[col].apply(normalize_text)

display(df_features[text_columns].head(5))


,name,domain,category,subcategory,brand,model,description,material,connection_type
0,chauffe-bain 11l junkers,chauffage,chaudière / générateur,chauffe-bain gaz,junkers,,chauffe-bain 11 l ; la fiche consultée indique...,,
1,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,chauffe-eau gaz,energical,smart 12l,chauffe-eau 12 l ; variantes de couleur affich...,,
2,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,chaudière gaz,energical,28 kw ventouse,chaudière ventouse annoncée avec puissance nom...,,
3,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-50,accélérateur référencé 99062433 ; puissance in...,,
4,accélérateur grundfos 15-60 110w réf. 99157090,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-60,accélérateur référencé 99157090 ; puissance in...,,


In [6]:
classification_columns = [
    col for col in ["domain", "category", "subcategory", "brand"]
    if col in df_features.columns
]

df_features["classification_text"] = (
    df_features[classification_columns]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\\s+", " ", regex=True)
    .str.strip()
)

display(df_features[["product_id", "classification_text"]].head())


,product_id,classification_text
0,SH001,chauffage chaudière / générateur chauffe-bain ...
1,SH002,chauffage chaudière / générateur chauffe-eau g...
2,SH003,chauffage chaudière / générateur chaudière gaz...
3,SH004,chauffage pompe de circulation circulateur / a...
4,SH005,chauffage pompe de circulation circulateur / a...


In [7]:
available_content_columns = [
    col for col in [
        "name", "domain", "category", "subcategory",
        "brand", "model", "description", "material", "connection_type"
    ]
    if col in df_features.columns
]

df_features["feature_text"] = (
    df_features[available_content_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\\s+", " ", regex=True)
    .str.strip()
)

display(df_features[["product_id", "feature_text"]].head())


,product_id,feature_text
0,SH001,chauffe-bain 11l junkers chauffage chaudière /...
1,SH002,chauffe-eau smart 12l energical chauffage chau...
2,SH003,chaudière ventouse 28 kw energical chauffage c...
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433 ...
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090...


In [8]:
for col in numeric_columns:
    df_features[col] = pd.to_numeric(df_features[col], errors="coerce")

numeric_summary = df_features[numeric_columns].describe().T
display(numeric_summary)


,count,mean,std,min,25%,50%,75%,max
diameter_mm,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
length_m,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pressure_rating_bar,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
power_kw,1.0,28.0,NaN,28.0,28.0,28.0,28.0,28.0
voltage_v,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_dzd,8.0,21237.5,8714.509658,1200.0,21300.0,22050.0,25125.0,29500.0


In [9]:
for col in numeric_columns:
    df_features[f"{col}_missing"] = df_features[col].isna().astype(int)

missing_flags = [f"{col}_missing" for col in numeric_columns]
display(df_features[["product_id"] + missing_flags].head())


,product_id,diameter_mm_missing,length_m_missing,pressure_rating_bar_missing,power_kw_missing,voltage_v_missing,price_dzd_missing
0,SH001,1,1,1,1,1,0
1,SH002,1,1,1,1,1,0
2,SH003,1,1,1,0,1,1
3,SH004,1,1,1,1,1,0
4,SH005,1,1,1,1,1,0


In [10]:
print("Identifiants manquants :", df_features["product_id"].isna().sum())
print("Identifiants dupliqués :", df_features["product_id"].duplicated().sum())
print("Textes descriptifs vides :", df_features["feature_text"].eq("").sum())

display(df_features[["product_id", "name", "feature_text"]])


Identifiants manquants : 0
Identifiants dupliqués : 0
Textes descriptifs vides : 0


,product_id,name,feature_text
0,SH001,chauffe-bain 11l junkers,chauffe-bain 11l junkers chauffage chaudière /...
1,SH002,chauffe-eau smart 12l energical,chauffe-eau smart 12l energical chauffage chau...
2,SH003,chaudière ventouse 28 kw energical,chaudière ventouse 28 kw energical chauffage c...
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433,accélérateur grundfos 15-50 75w réf. 99062433 ...
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090,accélérateur grundfos 15-60 110w réf. 99157090...
5,SH006,accélérateur grundfos 15-65 120w réf. 99062424,accélérateur grundfos 15-65 120w réf. 99062424...
6,SH007,accélérateur wilo 82w avec bloc pour chaudière...,accélérateur wilo 82w avec bloc pour chaudière...
7,SH008,accélérateur wilo 82w pour chaudière saunier d...,accélérateur wilo 82w pour chaudière saunier d...
8,SH009,adaptateur de robinet réglable,adaptateur de robinet réglable plomberie acces...


In [11]:
selected_columns = [
    col for col in [
        "product_id", "name", "domain", "category", "subcategory",
        "brand", "model", "price_dzd", "availability",
        "feature_text", "classification_text"
    ] + numeric_columns + missing_flags
    if col in df_features.columns
]

df_output = df_features[selected_columns].copy()

print("Dimensions du fichier de sortie :", df_output.shape)
display(df_output.head())


Dimensions du fichier de sortie : (9, 23)


,product_id,name,domain,category,subcategory,brand,model,price_dzd,availability,feature_text,...,pressure_rating_bar,power_kw,voltage_v,price_dzd,diameter_mm_missing,length_m_missing,pressure_rating_bar_missing,power_kw_missing,voltage_v_missing,price_dzd_missing
0,SH001,chauffe-bain 11l junkers,chauffage,chaudière / générateur,chauffe-bain gaz,junkers,,28500.0,In stock,chauffe-bain 11l junkers chauffage chaudière /...,...,NaN,NaN,NaN,28500.0,1,1,1,1,1,0
1,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,chauffe-eau gaz,energical,smart 12l,29500.0,In stock,chauffe-eau smart 12l energical chauffage chau...,...,NaN,NaN,NaN,29500.0,1,1,1,1,1,0
2,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,chaudière gaz,energical,28 kw ventouse,NaN,NaN,chaudière ventouse 28 kw energical chauffage c...,...,NaN,28.0,NaN,NaN,1,1,1,0,1,1
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-50,21800.0,NaN,accélérateur grundfos 15-50 75w réf. 99062433 ...,...,NaN,NaN,NaN,21800.0,1,1,1,1,1,0
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-60,21300.0,NaN,accélérateur grundfos 15-60 110w réf. 99157090...,...,NaN,NaN,NaN,21300.0,1,1,1,1,1,0


In [12]:
output_path = Path("../data/processed/products_features.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_output.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Fichier créé :", output_path.exists())
print("Chemin :", output_path)


Fichier créé : True
Chemin : ..\data\processed\products_features.csv


In [13]:
print("Produits :", len(df_output))
print("Features textuelles : feature_text, classification_text")
print("Features numériques :", numeric_columns)
print("Indicateurs de valeurs manquantes :", missing_flags)
display(df_output.head())


Produits : 9
Features textuelles : feature_text, classification_text
Features numériques : ['diameter_mm', 'length_m', 'pressure_rating_bar', 'power_kw', 'voltage_v', 'price_dzd']
Indicateurs de valeurs manquantes : ['diameter_mm_missing', 'length_m_missing', 'pressure_rating_bar_missing', 'power_kw_missing', 'voltage_v_missing', 'price_dzd_missing']


,product_id,name,domain,category,subcategory,brand,model,price_dzd,availability,feature_text,...,pressure_rating_bar,power_kw,voltage_v,price_dzd,diameter_mm_missing,length_m_missing,pressure_rating_bar_missing,power_kw_missing,voltage_v_missing,price_dzd_missing
0,SH001,chauffe-bain 11l junkers,chauffage,chaudière / générateur,chauffe-bain gaz,junkers,,28500.0,In stock,chauffe-bain 11l junkers chauffage chaudière /...,...,NaN,NaN,NaN,28500.0,1,1,1,1,1,0
1,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,chauffe-eau gaz,energical,smart 12l,29500.0,In stock,chauffe-eau smart 12l energical chauffage chau...,...,NaN,NaN,NaN,29500.0,1,1,1,1,1,0
2,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,chaudière gaz,energical,28 kw ventouse,NaN,NaN,chaudière ventouse 28 kw energical chauffage c...,...,NaN,28.0,NaN,NaN,1,1,1,0,1,1
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-50,21800.0,NaN,accélérateur grundfos 15-50 75w réf. 99062433 ...,...,NaN,NaN,NaN,21800.0,1,1,1,1,1,0
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-60,21300.0,NaN,accélérateur grundfos 15-60 110w réf. 99157090...,...,NaN,NaN,NaN,21300.0,1,1,1,1,1,0
